In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def fetch_cvrplib_bks_df():
    """
    爬取 CVRPLIB 的 BKS 数据并直接返回 pandas DataFrame。
    DataFrame 结构:
        Index: instance-name
        Column: bks (float)
    """
    target_url = "https://galgos.inf.puc-rio.br/cvrplib/index.php/en/bks_challenge/score/instances"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        print(f"正在请求: {target_url} ...")
        response = requests.get(target_url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # 定位表格
        table = soup.select_one('main section table')
        if not table:
            table = soup.find('table')
        
        if not table:
            print("错误: 未找到表格")
            return pd.DataFrame()

        # 提取数据
        data_list = []
        tbody = table.find('tbody')
        if tbody:
            rows = tbody.find_all('tr')
            for row in rows:
                cols = row.find_all('td')
                # 确保这一行至少有3列 (Index, Algo, Value)
                if len(cols) >= 3:
                    instance_name = cols[0].get_text(strip=True)
                    raw_value = cols[2].get_text(strip=True)

                    # --- 核心清洗逻辑 (参考你提供的文件) ---
                    try:
                        # 去除干扰字符：$, ,, {, }
                        clean_value = raw_value.replace('$', '').replace(',', '').replace('{', '').replace('}', '')
                        bks_value = float(clean_value)
                        
                        data_list.append({
                            'instance-name': instance_name,
                            'bks': bks_value
                        })
                    except ValueError:
                        continue # 无法转换则跳过

        # 构建 DataFrame
        if not data_list:
            return pd.DataFrame()

        df = pd.DataFrame(data_list)
        
        # 设置索引为 instance-name
        df.set_index('instance-name', inplace=True)
        
        return df

    except Exception as e:
        print(f"爬取过程中发生错误: {e}")
        return pd.DataFrame()

# ==========================================
# 运行并查看结果
# ==========================================
if __name__ == "__main__":
    df_bks = fetch_cvrplib_bks_df()
    
    if not df_bks.empty:
        print(f"成功爬取 {len(df_bks)} 条数据。")
        print("\n数据预览 (前 5 行):")
        print(df_bks.head())
        
        # 你的需求：只有一列 bks，横轴(Index)为 instance-name
        # df_bks 就是你要的结果
    else:
        print("未能获取数据。")

正在请求: https://galgos.inf.puc-rio.br/cvrplib/index.php/en/bks_challenge/score/instances ...
成功爬取 100 条数据。

数据预览 (前 5 行):
                    bks
instance-name          
XL-n1048-k237  380158.0
XL-n1094-k157  112431.0
XL-n1141-k112   95647.0
XL-n1188-k96   104407.0
XL-n1234-k55    96617.0


In [14]:
df_bks

,bks
instance-name,
XL-n1048-k237,380158.0
XL-n1094-k157,112431.0
XL-n1141-k112,95647.0
XL-n1188-k96,104407.0
XL-n1234-k55,96617.0
...,...
XL-n9160-k379,323691.0
XL-n9363-k209,205419.0
XL-n9571-k55,106365.0


In [9]:
import pandas as pd
import numpy as np

def process_optimization_results(file_path, sheet_name, name_mapping):
    """
    读取Excel，计算原始行最小值(BKS)，然后根据字典筛选和重命名列。
    """
    # 1. 读取数据 (本地请用 read_excel)
    # engine='openpyxl' 是读取 xlsx 的标准引擎
    df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # 2. 设置索引 (Instance Name)
    # 假设第一列是 Instance Name
    df.set_index(df.columns[0], inplace=True)
    df.index.name = 'instance-name'

    # 3. 数据清洗
    # 将 ' -', 空字符串等转为 NaN，并将数据转为数字类型
    def clean_and_convert(x):
        if isinstance(x, str):
            x = x.strip()
            if x in ['-', '']:
                return np.nan
        return x

    # 先清理字符
    df = df.applymap(clean_and_convert)
    # 再强制转为数字，无法转换的变为 NaN (errors='coerce')
    df = df.apply(pd.to_numeric, errors='coerce')

    # =======================================================
    # 新增步骤：计算最小值 (our_bks)
    # =======================================================
    # axis=1 表示横向计算（按行），min() 会自动忽略 NaN
    # 逻辑：在筛选列之前，先算好全表的最小值
    df['our_bks'] = df.min(axis=1)

    # 4. 筛选与重命名
    # 找出 df 中存在，且在 mapping 字典中也有定义的列
    valid_columns = [col for col in df.columns if col in name_mapping]
    
    if not valid_columns:
        print("警告: 字典没有匹配到任何列。")
        return pd.DataFrame()

    # 只保留：字典里的列 + 刚才算出来的 'our_bks'
    cols_to_keep = valid_columns + ['our_bks']
    df_processed = df[cols_to_keep].copy()

    # 重命名列
    df_processed.rename(columns=name_mapping, inplace=True)

    return df_processed


In [10]:
# ==========================================
# 用户配置区
# ==========================================

# 设置：输入文件路径（请确保文件在同目录下，或者是绝对路径）
file_path = 'latest_curves.xlsx'  # 你的原始 Excel 文件名

# 设置：你要读取的 Sheet 名称
sheet_name = 'Sheet2'  # 必须明确指定 Sheet 名字
# 2. 定义映射字典 {old_name: new_name}
# 只有在这个字典里出现的 key 才会保留在最终的 df 中
name_mapping = {
    'ails2_etaMax1.000_stoppingTime86400.00': '1day',
    'ails2_etaMax1.000_stoppingTime432000.00': '5day',
    'ails2_etaMax1.000_stoppingTime1296000.00': '15day',
    'ails2_etaMax1.000_stoppingTime864000.00': '10day',
    'ails2_etaMax1.000_stoppingTime1728000.00': '20day',
}

# ==========================================
# 执行处理
# ==========================================

df_result = process_optimization_results(file_path, sheet_name, name_mapping)

# 打印结果查看
print("处理后的 DataFrame 形状:", df_result.shape)
print(df_result.head())

# 如果需要，可以将结果保存为新的 Excel 或 CSV
# df_result.to_excel("processed_results.xlsx")

处理后的 DataFrame 形状: (100, 6)
                      1day       5day      10day  15day      20day    our_bks
instance-name                                                                
XL-n10001-k1570  2332950.0  2331541.0  2331315.0    NaN  2338657.0  2331201.0
XL-n1048-k237     380290.0   380224.0   380223.0    NaN   380386.0   380193.0
XL-n1094-k157     112456.0   112460.0   112451.0    NaN   112437.0   112433.0
XL-n1141-k112      95779.0    95755.0    95742.0    NaN    95746.0    95714.0
XL-n1188-k96      104466.0   104436.0   104424.0    NaN   104485.0   104416.0


C:\Users\10711\AppData\Local\Temp\ipykernel_35268\3226539990.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_and_convert)


In [11]:
df_result

,1day,5day,10day,15day,20day,our_bks
instance-name,,,,,,
XL-n10001-k1570,2332950.0,2331541.0,2331315.0,NaN,2338657.0,2331201.0
XL-n1048-k237,380290.0,380224.0,380223.0,NaN,380386.0,380193.0
XL-n1094-k157,112456.0,112460.0,112451.0,NaN,112437.0,112433.0
XL-n1141-k112,95779.0,95755.0,95742.0,NaN,95746.0,95714.0
XL-n1188-k96,104466.0,104436.0,104424.0,NaN,104485.0,104416.0
...,...,...,...,...,...,...
XL-n8960-k634,774280.0,NaN,NaN,NaN,NaN,773242.0
XL-n9160-k379,NaN,NaN,NaN,NaN,NaN,323784.0
XL-n9363-k209,205789.0,NaN,NaN,NaN,NaN,205601.0


In [19]:
def calculate_gaps(df_result, df_bks):
    """
    拼接结果表和BKS表，并计算所有列相对于官方BKS的Gap。
    Gap 公式: (Method_Value - Official_BKS) / Official_BKS
    """
    # 1. 拼接 (Join)
    # 使用 'left' 连接，以你的 Excel 结果(df_result)为主，保留你所有的 Instance
    # 爬虫没爬到的数据会自动填为 NaN
    df_combined = df_result.join(df_bks, how='left')

    # 检查是否有匹配失败的情况 (即爬虫有数据，但没匹配上，通常是名字空格问题)
    if df_combined['bks'].isna().all():
        print("警告: 拼接后 'bks' 列全为空。请检查两个表的 index (instance-name) 格式是否一致（如空格、大小写）。")

    # 2. 准备计算 Gap
    # 创建一个新的 DataFrame 用于存放 Gap
    df_gap = pd.DataFrame(index=df_combined.index)

    # 获取官方 BKS 列 (作为分母)
    official_bks = df_combined['bks']

    # 3. 遍历 df_result 的每一列进行计算
    # 这会包括你所有的算法列，以及 'our_bks' 列
    for col in df_result.columns:
        # 跳过非数值列（如果有的话）
        if not pd.api.types.is_numeric_dtype(df_result[col]):
            continue
            
        # 计算 Gap
        # 注意：如果 official_bks 是 NaN，结果自动为 NaN
        df_gap[col] = (df_combined[col] - official_bks) / official_bks * 100

    return df_combined, df_gap

In [20]:
df_all_values, df_gaps = calculate_gaps(df_result, df_bks)

In [21]:
df_gaps

,1day,5day,10day,15day,20day,our_bks
instance-name,,,,,,
XL-n10001-k1570,0.081337,0.020892,0.011197,NaN,0.326161,0.006306
XL-n1048-k237,0.034722,0.017361,0.017098,NaN,0.059975,0.009207
XL-n1094-k157,0.022236,0.025794,0.017789,NaN,0.005337,0.001779
XL-n1141-k112,0.138007,0.112915,0.099324,NaN,0.103506,0.070049
XL-n1188-k96,0.056510,0.027776,0.016282,NaN,0.074708,0.008620
...,...,...,...,...,...,...
XL-n8960-k634,0.169606,NaN,NaN,NaN,NaN,0.035318
XL-n9160-k379,NaN,NaN,NaN,NaN,NaN,0.028731
XL-n9363-k209,0.180120,NaN,NaN,NaN,NaN,0.088599
